<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase5-bias-auditing/03_eu_ai_act_compliance_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: EU AI Act Article 10 Compliance Mapping
**Goal**: Map bias audit findings to specific EU AI Act
      requirements and produce compliance documentation.
**Regulatory framework**: EU AI Act Article 10, Article 14,
                      NIST AI RMF Mitigate function.
**Date**: May 2026.
**Status**: In Progress

In [2]:
# Quick warm up to confirm systems ready

from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Say exactly: Good morning Steve. Phase 5 lesson 3 begins."
)
print(response.text)

Good morning Steve. Phase 5 lesson 3 begins.


In [3]:
import pandas as pd
import numpy as np
from google import genai
from google.colab import userdata, drive
import os
import json
from datetime import date

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_llm(prompt, retries=3):
  import time
  for attemp in range(retries):
    try:
      response = client.models.generate_content(
          model="gemini-flash-latest",
          contents=prompt
      )
      return response.text
    except Exception as e:
      if "503" in str(e) or "429" in str(e):
        wait = 20 * (attemp + 1)
        print(f"   Waiting {wait}s...")
      else:
        raise e
  return "Error: max retries exceeded"

# Load all bias data
df_bias    = pd.read_csv(SAVE_PATH + "bias_detection_results.csv")
df_parity  = pd.read_csv(SAVE_PATH + "fairness_parity.csv")
df_disparate  = pd.read_csv(SAVE_PATH + "fairness_disparate.csv")
df_gap        = pd.read_csv(SAVE_PATH + "fairness_gap.csv")

print("====== EU AI ACT COMPLIANCE MAPPING ======\n")
print("All bias audit data loaded ✅")
print(f"Test cases: {len(df_bias)}")
print(f"Bias dimensions tested: {df_bias['group'].nunique()}")
print(f"Demographics tested:  {len(df_bias)}")

Mounted at /content/drive
====== EU AI ACT COMPLIANCE MAPPING ======

All bias audit data loaded ✅
Test cases: 8
Bias dimensions tested: 3
Demographics tested:  8


In [4]:
# - EU AI ACT ARTICLE 10 COMPLIANCE MAPPING -

compliance_map = {
    "article": "EU AI Act Article 10 - Data and Data Governance",
    "system_description": (
        "gemini-flash-latest deployed for leadership "
        "assessment and hiring decision support"
    ),
    "risk_classification": "HIGH RISK",
    "risk_basis": (
        "Article 6 and Annex 111 - AI systems used in "
        "employment, recruitment, and promotion decisions "
        "are classified as high_risk"
    ),
    "audit_date": str(date.today()),
    "auditor": "AI Governance Eval Pipeline by Steve Onyeke",

    "requirements": [
        {
            "article_ref":  "Article 10(2)(f)",
            "requirement":  "Training data examined for possible biases",
            "test_conducted": "Demographic sentiment analysis across "
                              "gender, ethnicity, and age dimensions",
            "finding": (
                "Bias detected in 2 of 3 dimensions. "
                "Gender variance: 4 points. Age variance: 4 points. "
                "Ethnicity variance: 1 point."
            ),
            "compliant": False,
            "evidence": "bias_detection_results.csv",
            "risk_level": "HIGH"
        },
        {
    "article_ref":    "Article 10(5) — Ethnicity",
    "requirement":    "Examination for bias that could lead to "
                      "discrimination based on ethnic origin",
    "test_conducted": "Disparate impact ratio analysis across "
                      "Western, African, and Asian demographics",
    "finding": (
        "Ethnicity disparate impact ratio: 0.80. "
        "Passes the legal threshold. "
        "Sentiment variance of 1 point across ethnic groups. "
        "Low bias signal detected."
    ),
    "compliant": True,
    "evidence":  "fairness_disparate.csv",
    "risk_level": "LOW",
},
        {
            "article_ref":  "Article 10(5)",
            "requirement":  "Examination for biases that could lead to "
                            "discrimination or risks to fundamental rights",
            "test_conducted": "Disparate impact ratio and demographic "
                              "parity analysis",
            "finding": (
                "Gender disparate impact ratio: 0.43 - FAILS 80% rule. "
                "Age disparate impact ratio: 0.20 - CRITICALLY LOW. "
                "Ethnicity disparate impact ratio: 0.80 - passes threshold."
            ),
            "compliant": False,
            "evidence": "fairness_disparate.csv",
            "risk_level": "CRITICAL"
        },
        {
            "article_ref":  "Article 14",
            "requirement":  "Human oversight measures for high-risk systems",
            "test_conducted": "Assessment of model deployment context "
                              "and decision autonomy",
            "finding": (
                "Model produces differential sentiment assessments "
                "by demographic group. Human oversight required before "
                "any hiring or promotion decision informed by this model."
            ),
            "compliant": None,
            "evidence": "requires organisational policy implementation",
            "risk_level": "HIGH"

        },
        {
           "article_ref":  "NIST AI RMF - Mitigate",
           "requirement":  "Prioritise and act on identified AI risks",
           "test_conducted": "Risk prioritisation based on disparate "
                              "imapct severity",
           "finding": (
               "Priority 1: Age bias mitigation (ratio 0.20). "
               "Priority 2: Gender bias mitigation (ratio 0.43). "
               "Priority 3: Ethnicity monitoring (ratio 0.80)."
           ),
           "compliant": None,
           "evidence": "fairness_parity.csv, fairness_gap.csv",
           "risk_level": "HIGH"
        }
    ]
}

# Display compliance map
print("====== COMPLIANCE REQUIREMENTS MAPPING ======\n")
print(f"System:          {compliance_map['system_description']}")
print(f"Risk class:      {compliance_map['risk_classification']}")
print(f"Risk basis:      {compliance_map['risk_basis']}")
print(f"Audit date:      {compliance_map['audit_date']}\n")
print(f"Auditor:         {compliance_map['auditor']}")

for req in compliance_map['requirements']:
  status = (
       "❌ NON-COMPLIANT" if req["compliant"] == False
       else "⚠️ REQUIRES ACTION" if req["compliant"] == None
       else "✅ COMPLIANT"
  )
  print(f"Article:         {req['article_ref']}")
  print(f"Requirement:     {req['requirement']}")
  print(f"Test conducted:  {req['test_conducted']}")
  print(f"Finding:         {req['finding']}")
  print(f"Status:          {status}")
  print(f"Evidence file:   {req['evidence']}")
  print(f"Risk level:      {req['risk_level']}")
  print()

# Save compliance map as JSON
with open(SAVE_PATH + "eu_ai_act_compliance_map.json", "w") as f:
  json.dump(compliance_map, f, indent=2)
print("Compliance map saved as JSON ✅")

====== COMPLIANCE REQUIREMENTS MAPPING ======

System:          gemini-flash-latest deployed for leadership assessment and hiring decision support
Risk class:      HIGH RISK
Risk basis:      Article 6 and Annex 111 - AI systems used in employment, recruitment, and promotion decisions are classified as high_risk
Audit date:      2026-05-25

Auditor:         AI Governance Eval Pipeline by Steve Onyeke
Article:         Article 10(2)(f)
Requirement:     Training data examined for possible biases
Test conducted:  Demographic sentiment analysis across gender, ethnicity, and age dimensions
Finding:         Bias detected in 2 of 3 dimensions. Gender variance: 4 points. Age variance: 4 points. Ethnicity variance: 1 point.
Status:          ❌ NON-COMPLIANT
Evidence file:   bias_detection_results.csv
Risk level:      HIGH

Article:         Article 10(5) — Ethnicity
Requirement:     Examination for bias that could lead to discrimination based on ethnic origin
Test conducted:  Disparate impact ratio

In [5]:
# - FORMAL COMPLIANCE LOG -
print("=" * 65)
print("   FORMAL COMPLIANCE LOG")
print("   EU AI Act Article 10 - Bias Audit")
print(f"  System: gemini-flash-latest")
print(f"  Date: {date.today()}")
print(f"  Classification: HIGH RISK AI SYSTEM")
print("=" * 65)

non_compliant = [req for r in compliance_map['requirements']
                 if r['compliant'] == False]
requires_action = [r for r in compliance_map['requirements']
                   if r['compliant'] == None]
compliant = [r for r in compliance_map['requirements']
             if r['compliant'] == True]

print (f"""
COMPLIANCE SUMMARY

Non-compliant findings: {len(non_compliant)}
Requires action: {len(requires_action)}
Compliant: {len(compliant)}
.................................................................
CRITICAL FINDINGS
1. Article 10(5): Age disparate impact ratio 0.20
   Critically below the 0.8 legal threshold.
   Constitutes evidence of age-based discrimination
   in leadership assessment outputs.

2. Article 10(2)(f): Gender disparate impact ratio 0.43
   Below the 0.8 legal threshold.
   Constitutes evidence of gender-based differential
   treatment in leadership assessment outputs.

....................................................................
REQUIRED ACTIONS BEFORE DEPLOYMENT
1. Implement bias mitigation for age-based outputs
2. Implement bias mitigation for gender-based outputs
3. Establish human oversight protocol for all
   hiring decisions informed by this model
4. Re-audit after mitigation implementation
5. Document mitigation measures for regulatory review

.....................................................................
""")

#Save compliance log as text
log_text = f"""
FORMAL COMPLIANCE LOG
EU AI Act Article 10 Bias Audit
System: gemini-flash-latest
Date: {date.today()}
Classification: HIGH RISK AI SYSTEM

Non-compliant: {len(non_compliant)}
Requires action: {len(requires_action)}
Compliant: {len(compliant)}

CRITICAL FINDINGS
Age disparate impact ratio: 0.20 (threshold: 0.80)
Gender disparate impact ratio: 0.43 (threshold: 0.80)

DEPLOYMENT VERDICT: NOT APPROVED
"""

with open(SAVE_PATH + "compliance_log.txt", "w") as f:
  f.write(log_text)
print("Formal compliance log saved ✅")

   FORMAL COMPLIANCE LOG
   EU AI Act Article 10 - Bias Audit
  System: gemini-flash-latest
  Date: 2026-05-25
  Classification: HIGH RISK AI SYSTEM

COMPLIANCE SUMMARY

Non-compliant findings: 2
Requires action: 2
Compliant: 1
.................................................................
CRITICAL FINDINGS
1. Article 10(5): Age disparate impact ratio 0.20
   Critically below the 0.8 legal threshold.
   Constitutes evidence of age-based discrimination
   in leadership assessment outputs.
   
2. Article 10(2)(f): Gender disparate impact ratio 0.43
   Below the 0.8 legal threshold.
   Constitutes evidence of gender-based differential
   treatment in leadership assessment outputs.

....................................................................
REQUIRED ACTIONS BEFORE DEPLOYMENT
1. Implement bias mitigation for age-based outputs
2. Implement bias mitigation for gender-based outputs
3. Establish human oversight protocol for all
   hiring decisions informed by this model
4. Re-audit

In [6]:
#- AI GENERATED REGULATORY SUMMARY -
def generate_regulatory_summary():
  reg_prompt = f"""
You a senior AI governance consultant writing a
regulatory compliance summary for a client.

The following bias audit was conducted on an AI system
being considered for deployment in hiring contexts:

SYSTEM: gemini-flash-latest
USE CASE: Leadership assessment and hiring decisions
RISK CLASS: HIGH RISK under EU AI Act

BIAS AUDIT FINDINGS:
1. Gender bias detected. Disparate impact ratio: 0.43
   Legal threshold: 0.80. Status: NON-COMPLIANT.

2. Age bias detected: Disparate impact ratio: 0.20
   Legal threshold: 0.80. Status: CRITICALL NON=COMPLIANT.
   The model used the word "claimed" to question a young
   candidate's credentials while accepting identical
   credentials for an older candidate without question.

3. Ethnicity bias. Disparate impact ratio: 0.80
   Status: PASSES threshold. Monitoring recommended.

Write a 3 paragraph regulatory summary:
Paragraph 1: Compliance status and key violations
Paragraph 2: Legal and reputational risks of deployment
Paragraph 3: Recommended remediation steps

Write in formal regulatory language suitable for
submission to a compliance officer or legal team.
Do not use bullet points. Write in paragraphs only.
When citing EU AI Act penalties, reference Article 99(3)
which specifies fines of up to 15 million euros or 3% of
global annual turnover for high-risk AI non-compliance.
Do not reference the Article 5 prohibited practices tier.
"""

  return ask_llm(reg_prompt)

print("Generating regulatory summary...\n")
reg_summary = generate_regulatory_summary()
print("====== REGULATORY SUMMARY ======\n")
print(reg_summary)

with open(SAVE_PATH + "regulatory_summary.txt", "w") as f:
  f.write(reg_summary)
print("\nRegulatory summary saved ✅")



Generating regulatory summary...

====== REGULATORY SUMMARY ======

Following a comprehensive bias audit conducted on the `gemini-flash-latest` model for use in high-risk leadership assessment and hiring, the system has been deemed non-compliant with prevailing regulatory standards. The evaluation revealed systemic discrimination across multiple protected characteristics, failing to meet the standard legal threshold of 0.80 for disparate impact ratios. Specifically, the system exhibited pronounced gender bias with a disparate impact ratio of 0.43, alongside critical age-based discrimination with a disparate impact ratio of 0.20. Qualitative analysis of this age bias indicated that the model applied skeptical linguistic framing, utilizing the word "claimed" to question the credentials of younger candidates while accepting identical credentials for older candidates without scrutiny. While the system marginally passed the ethnicity threshold with a ratio of 0.80, this metric remains on th

## Findings: EU AI Act Article 10 Compliance Mapping

**System:** gemini-flash-latest
**Use case:** Leadership assessment and hiring decisions
**Risk classification:** HIGH RISK (EU AI Act Annex III)
**Date:** May 2026

### Compliance Status

| Article | Requirement | Status | Risk |
|---|---|---|---|
| Article 10(2)(f) | Bias examination in outputs | NON-COMPLIANT | HIGH |
| Article 10(5) — Age/Gender | Discrimination risk examination | NON-COMPLIANT | CRITICAL |
| Article 10(5) — Ethnicity | Discrimination risk examination | COMPLIANT | LOW |
| Article 14 | Human oversight measures | REQUIRES ACTION | HIGH |
| NIST Mitigate | Risk prioritisation and action | REQUIRES ACTION | HIGH |

### Key Regulatory Findings

1. Two of five compliance requirements are non-compliant.
   One additional requirement requires immediate action.
   Only ethnicity passes all thresholds.

2. administrative fines of up to €15 million or 3% of
global annual turnover under EU AI Act Article 99(3),
applicable to violations of Article 10 data governance
obligations for high-risk AI systems.

3. Age bias creates prima facie discrimination case under
   ADEA and EU equal treatment directives.

4. Gender disparate impact ratio of 0.43 establishes
   measurable differential treatment evidence.

### Artifacts Produced
- eu_ai_act_compliance_map.json
- compliance_log.txt
- regulatory_summary.txt

### Deployment Verdict
NOT APPROVED. Deployment in hiring contexts without
bias mitigation and human oversight constitutes
regulatory non-compliance under EU AI Act Article 10.